In [22]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus
/home/ahmed-el-kaffas/Documents/Github/QuantUS/engines


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [23]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'mp4', 'nifti']


In [24]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/Standford/SHC-P09/V01/SHC-P09-V01-CE1_18.33.17_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/Standford/SHC-P09/V01/SHC-P09-V01-CE1_18.33.17_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [25]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [8]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [9]:
seg_type = 'nifti'

seg_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V03/UCSD-P07-V03-CE1-MC_VOI.nii.gz'
seg_loader_kwargs = {}

In [10]:
from src.entrypoints import seg_loading_step

# Testing the motion compensation, right now is hard coded
seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Figure 1 Display the motion compensation from B mode and CEUS


Figure 1: Axial Plane of the 3D contrast enhanced ultrasound and B mode in 5 consecutive frames. Top row: no motion compensation. Bottom row: motion compensated data. The red dash line represents the boarder of the region of interests


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import binary_erosion

def get_mask_boundary(mask_slice):
    if mask_slice.max() == 0:
        return np.zeros_like(mask_slice, dtype=bool)
    eroded = binary_erosion(mask_slice)
    return mask_slice.astype(bool) & ~eroded

def get_voi_center(mask_3d):
    coords = np.where(mask_3d > 0)
    if len(coords[0]) == 0:
        return None, None, None
    return (int(np.mean(coords[0])),   # lateral  X
            int(np.mean(coords[1])),   # depth    Y
            int(np.mean(coords[2])))   # elevation Z
def enhance_bmode_noise(image_slice, p_low_percentile=15.0, p_high_percentile=98.5):
    non_zero = image_slice[image_slice != 0]
    p_low = np.percentile(non_zero, p_low_percentile)
    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(image_slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

def apply_clahe(img_u8, clip=2.0, grid=8):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(img_u8)

In [ ]:
from scipy.ndimage import shift as ndshift

# --- voxel spacing (mm). extras_dict stores it as (z, y, x) ---
# volume axes are (X=lateral, Y=depth, Z=elevation, T)
sz, sy, sx = image_data.pixdim   # z, y, x  in mm

nx, ny, nz, num_frames = image_data.pixel_data.shape

show_frames = [20, 21, 22, 23, 24]
ticks = [0, 20, 40, 60, 80]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = bmode_image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_bmode_noise(raw, p_low_percentile=5.0, p_high_percentile=99.5) 
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_bmode_noise(raw_mc, p_low_percentile=5.0, p_high_percentile=99.5)
    # img_mc = apply_clahe(img_mc.astype(np.uint8), clip=3, grid=8) 
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

In [ ]:
# CEUS images enhance with noise reduction from first frame
def compute_ceus_noise_floor(first_frame) -> float:
    """
    Compute the noise floor (p_low scalar) from pre-contrast frames of a 4D CEUS scan.
    Returns the noise floor value as a float.
    """
    pixel_data = first_frame

    ref_frames = pixel_data
    ref_nonzero = ref_frames[ref_frames != 0]
    if ref_nonzero.size == 0:
        raise ValueError("Pre-contrast reference frames contain no non-zero values.")

    noise_mean = np.mean(ref_nonzero)
    noise_std = np.std(ref_nonzero)
    return float(noise_mean+noise_std)  # noise floor is mean + std

def enhance_ceus(slice, p_low, p_high_percentile=99.5):
    """
    Enhance a single CEUS image slice using noise floor and high percentile.
    Returns the enhanced image slice.
    """
    non_zero = slice[slice != 0]
    if non_zero.size == 0:
        return np.zeros_like(slice, dtype=np.uint8)

    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

In [ ]:
show_frames = [20, 21, 22, 23, 24]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

ticks = [0, 20, 40, 60, 80]

# Compute baseline noise floor from the first frame of the CEUS scan
first_frame = image_data.pixel_data[:, :, :, 0]*seg_data.seg_mask
noise_floor = compute_ceus_noise_floor(first_frame)*1.5

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_ceus(raw, p_low=noise_floor, p_high_percentile=99.5)
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_ceus(raw_mc, p_low=noise_floor, p_high_percentile=99.5)
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

# Figure 4: Parameteric Map analysis

This script only works once finish the parametric analysis get the paramaps_LOGNORMAL. NPY files for different parameters

Figure 4: Parametric maps of 3D DCE analysis using motion compensation and without using motion compensation

In [ ]:
# ============================================================================
# MATPLOTLIB 2D VISUALIZATION
# ============================================================================
from typing import Optional, List
import os
import numpy as np

def orient_camera(p, grid, zoom=1.0):
    """Set X-horizontal, Y-vertical, Z-into-plane orientation.
    zoom < 1.0 zooms out, > 1.0 zooms in."""
    cx, cy, cz = grid.center
    xmin, xmax, ymin, ymax, zmin, zmax = grid.bounds
    span = max(xmax - xmin, ymax - ymin)
    dist = span * 2.5 / zoom          # smaller zoom -> larger dist -> further out
    p.camera_position = [
        (cx, cy, zmax - dist),
        (cx, cy, cz),
        (0.0, -1.0, 0.0),
    ]
    p.reset_camera_clipping_range()
    p.camera.zoom(zoom)               # also scales the view


def save_sweep_video(p, grid, out_dir, filename="auc_sweep.mp4",
                     total_deg=180.0, n_frames=90, framerate=20):
    """Rotate the camera through `total_deg` in azimuth and write a movie."""
    orient_camera(p, grid)
    step = total_deg / n_frames

    path = os.path.join(out_dir, filename)
    p.open_movie(path, framerate=framerate)
    p.write_frame()                       # first frame at starting orientation
    for _ in range(n_frames):
        p.camera.azimuth += step          # rotate around view-up (Y) axis
        p.reset_camera_clipping_range()
        p.write_frame()
    print(f"Saved video: {path}")
    return path


def save_sweep_frames(p, grid, out_dir, prefix="auc_frame",
                      total_deg=180.0, n_frames=90, save_every=18, scale=2,zoom=1.0):
    """Rotate through `total_deg` and save a screenshot every `save_every` frames."""
    orient_camera(p, grid,zoom=zoom)
    step = total_deg / n_frames

    saved = []
    for i in range(n_frames + 1):         # include frame 0
        if i > 0:
            p.camera.azimuth += step
            p.reset_camera_clipping_range()
        if i % save_every == 0:
            path = os.path.join(out_dir, f"{prefix}_{i:03d}.png")
            p.screenshot(path, scale=scale)
            saved.append(path)
            print(f"Saved frame: {path}")
    return saved

In [26]:
import os
import time
import numpy as np
import napari
from skimage.morphology import remove_small_objects, binary_erosion, ball
from skimage.restoration import denoise_nl_means, estimate_sigma

# Loading the data
visualization_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc'
AUC        = np.load(f'{visualization_path}/AUC_full_TIC_numerical.npy')   # (X, Y, Z)
image      = np.load(f'{visualization_path}/image.npy')                   # (X, Y, Z, T) or (X, Y, Z)
pixel_size = np.load(f'{visualization_path}/pix_dims.npy')                # (dx, dy, dz) mm
dx, dy, dz = pixel_size

image = image_data.pixel_data


time_point = 15  # initial frame the viewer opens on
# --- Load the motion-compensated VOI first so we can center the view on it ---
seg = np.load(os.path.join(visualization_path, 'segmentation.npy'))
n_frames = image.shape[-1]
mc = globals().get('seg_data', None)
mc = getattr(mc, 'motion_compensation', None) if mc is not None else None
mc_available = (mc is not None and seg.shape == seg_data.seg_mask.shape
                 and len(mc.translation_vectors) == n_frames)

if mc_available:
    voi_4d = mc.apply_to_all_frames(seg, order=0)   # (X, Y, Z, T)
else:
    print("Warning: seg_data.motion_compensation doesn't match this video "
          "(different exam or frame count) -- showing the static reference "
          "VOI only, not motion-compensated.")
    voi_4d = np.repeat(seg[..., None], n_frames, axis=-1)  # (X, Y, Z, T), unshifted

def voi_center_z(mask_3d):
    """Elevation (Z) index through the middle of the VOI, or the volume's
    geometric mid-slice if the mask is empty at this frame."""
    coords = np.argwhere(mask_3d > 0)
    if len(coords) == 0:
        return mask_3d.shape[2] // 2
    return int(round(coords[:, 2].mean()))

z_center = voi_center_z(voi_4d[..., time_point])
print(f"Centering the Elevation (Z) slice on the VOI's middle: z={z_center}")

# --- Reorder every layer to (T, Z, X, Y) ---
ceus_volume = np.transpose(image, (3, 2, 0, 1))[:, z_center:, :, :]   # (T, Z', X, Y)
voi_4d = np.transpose(voi_4d, (3, 2, 0, 1))[:, z_center:, :, :]       # (T, Z', X, Y)
auc_3d = np.transpose(AUC, (2, 0, 1))[z_center:, :, :]                # (Z', X, Y), no time axis

# --- Speckle denoising, per frame ---
t0 = time.time()
ceus_denoised = np.empty_like(ceus_volume, dtype=np.float32)
for t in range(ceus_volume.shape[0]):
    frame = ceus_volume[t].astype(np.float32)                      # (Z', X, Y)
    sigma_est = float(np.mean(estimate_sigma(frame, channel_axis=0)))
    ceus_denoised[t] = denoise_nl_means(
        frame, channel_axis=0,
        h=1.15 * sigma_est, sigma=sigma_est,
        patch_size=5, patch_distance=6, fast_mode=True,
    )
    if t % 10 == 0:
        print(f"  denoised frame {t}/{ceus_volume.shape[0]}")
print(f"NLM denoised {ceus_volume.shape[0]} frames in {time.time() - t0:.1f}s")
ceus_volume = ceus_denoised

Centering the Elevation (Z) slice on the VOI's middle: z=69
  denoised frame 0/364
  denoised frame 10/364
  denoised frame 20/364
  denoised frame 30/364
  denoised frame 40/364
  denoised frame 50/364
  denoised frame 60/364
  denoised frame 70/364
  denoised frame 80/364
  denoised frame 90/364
  denoised frame 100/364
  denoised frame 110/364
  denoised frame 120/364
  denoised frame 130/364
  denoised frame 140/364
  denoised frame 150/364
  denoised frame 160/364
  denoised frame 170/364
  denoised frame 180/364
  denoised frame 190/364
  denoised frame 200/364
  denoised frame 210/364
  denoised frame 220/364
  denoised frame 230/364
  denoised frame 240/364
  denoised frame 250/364
  denoised frame 260/364
  denoised frame 270/364
  denoised frame 280/364
  denoised frame 290/364
  denoised frame 300/364
  denoised frame 310/364
  denoised frame 320/364
  denoised frame 330/364
  denoised frame 340/364
  denoised frame 350/364
  denoised frame 360/364
NLM denoised 364 frames in

In [27]:
import imageio.v3 as iio


def orient_napari_camera(viewer, angle_deg):
    """Point the 3D camera so the (Lateral, Depth) plane faces front at
    angle_deg=0 -- the same plane the 2D slice view above shows -- then
    sweep it around the Depth axis as angle_deg increases toward a side
    view, mirroring orient_camera()/camera.azimuth for the PyVista plot
    further up in this notebook.

    With ndisplay=3, viewer.dims.displayed is (Elevation Z, Lateral X,
    Depth Y), so the camera's view/up 3-tuples line up with that (Z, X, Y)
    order. Rotating the view direction from (1,0,0) toward (0,1,0) sweeps
    it from "looking down Elevation" (front) to "looking down Lateral"
    (side), while up stays fixed along -Depth -- an azimuth rotation about
    the Depth ("Y") axis, the same convention the PyVista sweep used with
    its up=(0,-1,0) (also Depth) and camera.azimuth increments.
    """
    theta = np.deg2rad(angle_deg)
    view_direction = (np.cos(theta), np.sin(theta), 0.0)
    up_direction = (0.0, 0.0, -1.0)
    viewer.camera.set_view_direction(view_direction, up_direction=up_direction)


def _napari_sweep_angles(angle_start, angle_stop, angle_step):
    n = int(round((angle_stop - angle_start) / angle_step)) + 1
    return [angle_start + i * angle_step for i in range(n)]


def _napari_set_time(viewer, time_point):
    viewer.dims.ndisplay = 3
    current_step = list(viewer.dims.current_step)
    current_step[0] = time_point   # axis 0 = Time, per viewer.dims.axis_labels
    viewer.dims.current_step = tuple(current_step)


def napari_sweep_frames(viewer, out_dir, time_point=15, prefix="ceus_sweep",
                         angle_start=0.0, angle_stop=90.0, angle_step=5.0,
                         scale=2):
    """Rotate the napari 3D camera from angle_start to angle_stop and save
    a screenshot at each angle_step. Mirrors save_sweep_frames() above."""
    _napari_set_time(viewer, time_point)

    saved = []
    for angle in _napari_sweep_angles(angle_start, angle_stop, angle_step):
        orient_napari_camera(viewer, angle)
        path = os.path.join(out_dir, f"{prefix}_{angle:05.1f}deg.png")
        viewer.screenshot(path, scale=scale)
        saved.append(path)
        print(f"Saved frame: {path} (angle={angle:.1f} deg)")
    return saved


def napari_sweep_video(viewer, out_dir, filename="ceus_sweep.mp4",
                        time_point=15, angle_start=0.0, angle_stop=90.0,
                        angle_step=5.0, framerate=10, scale=2):
    """Same sweep as napari_sweep_frames(), written to an mp4 instead of
    individual PNGs. napari has no p.open_movie() equivalent, so frames are
    captured to in-memory arrays and encoded with imageio (already a napari
    dependency) rather than written to disk one by one."""
    _napari_set_time(viewer, time_point)

    frames = []
    for angle in _napari_sweep_angles(angle_start, angle_stop, angle_step):
        orient_napari_camera(viewer, angle)
        frames.append(viewer.screenshot(scale=scale))

    path = os.path.join(out_dir, filename)
    iio.imwrite(path, frames, fps=framerate)
    print(f"Saved video: {path} ({len(frames)} frames)")
    return path


# Frame #15, (Lateral, Depth) plane front-on, sweeping 0-90 deg around the
# Depth ("Y") axis in 5 deg steps:
# napari_sweep_frames(viewer, visualization_path, time_point=15,
#                      angle_start=0.0, angle_stop=90.0, angle_step=5.0)


In [32]:
raw_image = np.transpose(image, (3, 2, 0, 1))
raw_image.shape

(364, 341, 458, 338)

In [ ]:
# Napari viewer
viewer = napari.Viewer()
# Tune `1.15 * sigma_est` up for stronger smoothing, down to keep more texture.
ceus_layer = viewer.add_image(
    image_data.pixel_data[:,100:,:,:],
    name='CEUS',
    colormap='gray',
    opacity=1.0,
    gamma = 1.3,
    rendering='attenuated_mip'
)

# T0 map overlay: static (Z, X, Y), no time axis -- napari broadcasts a
# lower-dimensional layer across the leading T slider automatically, so this
# stays fixed while the CEUS video and VOI scrub through time.
valid_vals = AUC[~np.isnan(AUC)]
if len(valid_vals) > 0:
    vmin, vmax = np.percentile(valid_vals, [5, 95])
    viewer.add_image(
        auc_3d,
        name="T0",
        colormap='turbo',
        contrast_limits=[vmin, vmax],
        opacity=0.4,
        rendering='minip'
    )

# viewer.add_labels(
#     voi_4d.astype(int),
#     name='MC VOI' if mc_available else 'VOI (static, not motion-compensated)',
#     opacity=0.4,
# )

# --- Orientation cues, PyVista-style: labeled XYZ triad + a bounding box ---
# viewer.axes is napari's orientation triad (like PyVista's add_axes widget).
# There's no direct per-axis color field -- `colored` is a boolean: True gives
# cyan/yellow/magenta per axis, False collapses all axes to one color, the
# inverse of the canvas background. The default 'dark' theme's canvas is pure
# black, so colored=False renders the triad in solid white. The bounding_box
# overlay lives per-layer (not on the viewer), so it's attached to the CEUS
# layer -- it draws a wireframe box around that layer's current data extent,
# the "box" part PyVista's show_grid() gives you.
# Note: in ndisplay=2 (current mode) both only show the 2 axes actually being
# displayed (Lateral X, Depth Y) -- Elevation (Z) is a slider dim right now,
# not a rendered one, so it won't appear as a third arrow/edge unless you
# switch to viewer.dims.ndisplay = 3.
viewer.axes.visible = True
viewer.axes.labels = True
viewer.axes.colored = False
ceus_layer.bounding_box.visible = True
ceus_layer.bounding_box.line_color = 'white'
ceus_layer.bounding_box.line_thickness = 1.5

viewer.dims.axis_labels = ['Time', 'Elevation (Z)', 'Lateral (X)', 'Depth (Y)']
viewer.dims.ndisplay = 2           # 2D slice view, not a 3D volume blend
# Z=0 in the truncated arrays *is* z_center now, so the slider starts at 0.
viewer.dims.current_step = (time_point, 0, 0, 0)
viewer.title = 'CEUS Perfusion with T0 Map and Motion-Compensated VOI'

In [ ]:
napari_sweep_video(viewer, visualization_path, time_point=15,
                    angle_start=0.0, angle_stop=90.0, angle_step=1)

/tmp/ipykernel_622561/2296839327.py:65: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  orient_napari_camera(viewer, angle)
/home/ahmed-el-kaffas/Documents/Github/QuantUS/.venv/lib/python3.12/site-packages/napari/_vispy/camera.py:203: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  self.angles = self._camera.angles
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2716, 1568) to (2720, 1568) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved video: /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc/ceus_sweep.mp4 (181 frames)


'/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc/ceus_sweep.mp4'